In [ ]:
pip install Deepface

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 2.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.6/108.6 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 37.7 MB/s eta 0:00:00
  Created wheel for fire: filename=fire-0.7.0-py3-none-any.whl size=114249 sha256=b3d72baa42b82d2e0a87d249b6018ae00ffb313880f37a7982b081c1214940c6
  Stored in directory: /root/.cache/pip/wheels/19/39/2f/2d3cadc408a8804103f1c34ddd4b9f6a93497b11fa96fe738e
Successfully built fire


In [ ]:
import cv2
import numpy as np
from deepface import DeepFace
from google.colab.output import eval_js
from IPython.display import display, Javascript
from base64 import b64decode, b64encode
import os

# Load Haar Cascade for face detection
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

# Path to your images directory
db_path = "/content/images"  # Ensure this path is correct

# Extract the person's name from the folder path
def get_person_name_from_path(identity_path):
    folder_name = os.path.basename(os.path.dirname(identity_path))
    return folder_name

# Recognize the face and return the name, confidence, and color
def recognize_face(face, db_path):
    try:
        # Save the cropped face temporarily
        face_filename = 'temp_face.jpg'
        cv2.imwrite(face_filename, face)

        # Find matches in the database
        matches_list = DeepFace.find(img_path=face_filename, db_path=db_path, model_name="Facenet", enforce_detection=False)

        if matches_list and not matches_list[0].empty:
            best_match = matches_list[0].iloc[0]
            identity_path = best_match["identity"]
            name = get_person_name_from_path(identity_path)
            distance = best_match.get("distance", None)
            confidence = max(0, 100 - (distance * 100)) if distance and distance < 0.15 else 0

            if confidence < 75:
                name, confidence = "Unknown", 0
                color = (0, 0, 255)  # Red for unknown faces
            else:
                color = (0, 255, 0)  # Green for known faces
        else:
            name, confidence = "Unknown", 0
            color = (0, 0, 255)

    except Exception as e:
        print(f"Error: {e}")
        name, confidence, color = "Unknown", 0, (0, 0, 255)

    return name, confidence, color

# JavaScript code to capture a frame from webcam
def video_stream():
    js = Javascript('''
        async function captureFrame() {
            const video = document.createElement('video');
            const stream = await navigator.mediaDevices.getUserMedia({ video: true });

            video.srcObject = stream;
            await video.play();

            const canvas = document.createElement('canvas');
            canvas.width = video.videoWidth;
            canvas.height = video.videoHeight;

            const context = canvas.getContext('2d');
            context.drawImage(video, 0, 0);

            stream.getVideoTracks()[0].stop();
            return canvas.toDataURL('image/jpeg');
        }
        captureFrame();
    ''')
    display(js)

# Convert JavaScript image data to OpenCV format
def js_to_image(js_reply):
    image_bytes = b64decode(js_reply.split(',')[1])
    jpg_as_np = np.frombuffer(image_bytes, dtype=np.uint8)
    img = cv2.imdecode(jpg_as_np, flags=1)
    return img

# Function to continuously capture and process frames
def process_stream():
    try:
        while True:
            # Capture a frame from the video stream
            js_reply = eval_js('captureFrame()')
            frame = js_to_image(js_reply)

            # Convert to grayscale for face detection
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

            # Detect faces in the frame
            faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5)

            # Iterate over detected faces
            for (x, y, w, h) in faces:
                face = frame[y:y+h, x:x+w]
                name, confidence, color = recognize_face(face, db_path)

                # Draw bounding box and label
                cv2.rectangle(frame, (x, y), (x+w, y+h), color, 2)
                cv2.putText(frame, f"{name} ({confidence:.2f}%)", (x, y - 10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.9, color, 2)

            # Display the processed frame
            _, buffer = cv2.imencode('.jpg', frame)
            frame_data = b64encode(buffer).decode('utf-8')

            display(Javascript(f"""
                const img = new Image();
                img.src = 'data:image/jpeg;base64,{frame_data}';
                document.body.appendChild(img);
            """))

    except KeyboardInterrupt:
        print("Streaming stopped.")

# Start the video stream and process frames
video_stream()
process_stream()
